In [6]:
# =============================================================================
# SHIELD — REPEATED NESTED CROSS-VALIDATION ON MODMA (WITHIN-DATASET)
# =============================================================================
# Purpose  : same-dataset, subject-level comparison with prior MODMA studies
# Dataset  : MODMA (expected n = 53: 24 MDD + 29 HC), same 314 features as PGIMER
# Pipeline : identical to the PGIMER analysis
#            RobustScaler -> ANOVA SelectKBest -> classifier, all inside a
#            Pipeline; inner LOOCV selects k on the outer training fold only
# Changes vs PGIMER script:
#   1. MODMA data path and output file names
#   2. Explicit, checked label coding (MDD = 1, HC = 0)
#   3. Feature columns checked against the PGIMER feature list (optional)
#   4. Zero-variance features removed and reported instead of stopping
#   5. No z-score step (RobustScaler inside the pipeline handles scaling)
#   6. Biomarker ranking (old Cell 13) removed — biomarkers come from PGIMER only
#   7. 1D-CNN added (same architecture as PGIMER; inner stratified 5-fold CV over k)
# Report SVM (primary) and 1D-CNN in Table 9; all models in supplementary.
# =============================================================================

# %%
# =============================================================================
# CELL 1 — IMPORTS & CONFIGURATION
# =============================================================================
import warnings
warnings.filterwarnings("ignore")

import os
import pickle
import numpy as np
import pandas as pd

from collections import Counter
from itertools import combinations

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold, LeaveOneOut, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)
from sklearn.utils import resample

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
    print("XGBoost is not installed. Run: pip install xgboost")

try:
    from statsmodels.stats.contingency_tables import mcnemar as mcnemar_test
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False
    print("statsmodels is not installed. Run: pip install statsmodels")

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv1D, MaxPooling1D, Flatten, Dense,
                                     Dropout, BatchNormalization, Reshape)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
tf.get_logger().setLevel("ERROR")

# ------------------------------
# Configuration — CHANGE PATHS
# ------------------------------
DATA_PATH  = "E:/final_eeg_data_MODMA_updated.csv"       # MODMA feature file
OUTPUT_DIR = "modma_within_nested_cv_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUT_PREFIX = os.path.join(OUTPUT_DIR, "MODMA_within")     # prefix for saved results

# ---- Column-name mapping: MODMA (128-channel indices) -> PGIMER (19-channel order) ----
MODMA_ELECTRODE_MAP = {
    'Fp2': 8,   'Fz':  10,  'Fp1': 21,  'F3':  23,
    'F7':  32,  'C3':  35,  'T3':  44,  'P3':  51,
    'T5':  57,  'Pz':  61,  'O1':  69,  'O2':  82,
    'P4':  91,  'T6':  95,  'C4':  103, 'T4':  107,
    'F8':  121, 'F4':  123, 'Cz':  128,
}
PRIVATE_ELECTRODE_ORDER = [
    'Fp1','F3','C3','P3','O1','F7','T3','T5',
    'Fp2','F4','C4','P4','O2','F8','T4','T6',
    'Fz','Cz','Pz'
]
FEATURE_TYPE_LIST = [
    'Alpha_Band_psd_mean',    'Beta_Band_psd_mean',
    'Gamma_Band_psd_mean',    'Theta_Band_psd_mean',
    'Delta_Band_psd_mean',    'Alpha_Band_psd_variance',
    'Beta_Band_psd_variance', 'Gamma_Band_psd_variance',
    'Theta_Band_psd_variance','Delta_Band_psd_variance',
    'Fractal_Dimension',      'Wavelet_Energy',
    'Spectral_Entropy',       'Hjorth_Activity',
    'Hjorth_Mobility',        'Hjorth_Complexity'
]
EXTRA_FEATURES = [
    'Alpha_Asymmetry_Fp2_minus_Fp1',
    'Alpha_Asymmetry_F4_minus_F3',
    'Alpha_Asymmetry_F8_minus_F7',
    'Theta_Alpha_Ratio_Fp1',
    'Theta_Alpha_Ratio_F3',
    'Theta_Alpha_Ratio_F4',
    'Theta_Alpha_Ratio_Fp2',
    'Theta_Alpha_Ratio_F7',
    'Theta_Alpha_Ratio_F8',
    'Theta_Alpha_Ratio_Fz'
]

LABEL_COL = "MDD"
DROP_COLS = ["MDD", "ID", "Species", "Subject", "subject_id"]   # add any other non-feature columns

K_VALUES      = [10, 15, 20, 25, 30]
N_OUTER_FOLDS = 10
OUTER_SEEDS   = [42, 0, 1, 7, 99]
N_BOOTSTRAP   = 1000
SEED          = 42

CLASSICAL = ["RF", "SVM_Lin", "LR", "LDA"] + (["XGBoost"] if HAS_XGBOOST else [])
RUN_CNN = True                      # set False to skip the 1D-CNN
CNN_INNER_FOLDS = 5                 # inner CV for k (LOOCV impractical for a CNN)
CLF_NAMES = CLASSICAL + (["1D-CNN"] if RUN_CNN else [])
METRICS   = ["accuracy", "precision", "sensitivity", "specificity", "f1", "roc_auc"]

print("=" * 80)
print("SHIELD — REPEATED NESTED CV WITHIN MODMA")
print("=" * 80)
print(f"Classifiers : {CLF_NAMES}")
print(f"k values    : {K_VALUES}")
print(f"Outer folds : {N_OUTER_FOLDS}   Seeds : {OUTER_SEEDS}")
print("=" * 80)


# %%
# =============================================================================
# CELL 2 — DATA LOADING AND CHECKS
# =============================================================================
df = pd.read_csv(DATA_PATH)
print(f"Raw shape: {df.shape}")

# ---- 1. Label coding: force MDD = 1, HC = 0 (no LabelEncoder sorting) ----
raw = df[LABEL_COL]
if raw.dtype == object:
    mapping = {"MDD": 1, "HC": 0, "mdd": 1, "hc": 0}
    unknown = set(raw.unique()) - set(mapping)
    assert not unknown, f"Unrecognised labels: {unknown} — extend the mapping"
    Y = raw.map(mapping).values.astype(int)
else:
    Y = raw.values.astype(int)
assert set(np.unique(Y)) == {0, 1}, "Labels must be binary 0/1"
print(f"Labels: MDD (1) = {np.sum(Y == 1)} | HC (0) = {np.sum(Y == 0)}  "
      f"(expected 24 MDD, 29 HC)")

# ---- 2. Feature matrix: rename MODMA columns to the PGIMER naming and order ----
def build_modma_feature_frame(df):
    """Map MODMA columns (<feature>_<128-ch index>) to PGIMER names
    (<feature>_<19-ch index>) in the PGIMER order: 16 types x 19 channels + 10 extras = 314."""
    data, missing = {}, []
    for ftype in FEATURE_TYPE_LIST:
        for priv_idx, electrode in enumerate(PRIVATE_ELECTRODE_ORDER):
            modma_col = f"{ftype}_{MODMA_ELECTRODE_MAP[electrode]}"
            priv_col = f"{ftype}_{priv_idx}"
            if modma_col in df.columns:
                data[priv_col] = df[modma_col].values
            else:
                missing.append(modma_col)
    for col in EXTRA_FEATURES:                      # same names in both files
        if col in df.columns:
            data[col] = df[col].values
        else:
            missing.append(col)
    assert not missing, (f"{len(missing)} expected MODMA columns not found, e.g. {missing[:5]}. "
                         "Check the column naming in the MODMA CSV.")
    return pd.DataFrame(data)

X_df = build_modma_feature_frame(df)
assert X_df.shape[1] == len(FEATURE_TYPE_LIST) * len(PRIVATE_ELECTRODE_ORDER) + len(EXTRA_FEATURES) == 314
print(f"Mapped MODMA features to PGIMER naming: {X_df.shape[1]} features")

X = X_df.values.astype(float)
feature_names = list(X_df.columns)
assert not np.isnan(X).any(), "X contains NaN values"
assert not np.isinf(X).any(), "X contains infinite values"

# ---- 4. Remove zero-variance features (report, do not stop) ----
zero_var = np.where(X.var(axis=0) == 0)[0]
if len(zero_var):
    print(f"Removing {len(zero_var)} zero-variance features: "
          f"{[feature_names[i] for i in zero_var][:10]}")
    keep = np.setdiff1d(np.arange(X.shape[1]), zero_var)
    X = X[:, keep]
    feature_names = [feature_names[i] for i in keep]
print(f"Final feature matrix: X = {X.shape}, Y = {Y.shape}")


# %%
# =============================================================================
# CELL 3 — METRIC FUNCTIONS
# =============================================================================
def compute_metrics(y_true, y_pred, y_prob):
    y_true, y_pred, y_prob = map(np.asarray, (y_true, y_pred, y_prob))
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy":    accuracy_score(y_true, y_pred),
        "precision":   precision_score(y_true, y_pred, zero_division=0),
        "sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        "f1":          f1_score(y_true, y_pred, zero_division=0),
        "roc_auc":     roc_auc_score(y_true, y_prob),
        "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn),
    }


def bootstrap_ci(y_true, y_pred, y_prob, n_bootstrap=N_BOOTSTRAP):
    y_true, y_pred, y_prob = map(np.asarray, (y_true, y_pred, y_prob))
    res = {m: [] for m in METRICS}
    for i in range(n_bootstrap):
        idx = resample(np.arange(len(y_true)), replace=True,
                       stratify=y_true, random_state=i)
        yt, yp, yb = y_true[idx], y_pred[idx], y_prob[idx]
        if len(np.unique(yt)) < 2:
            continue
        m = compute_metrics(yt, yp, yb)
        for key in METRICS:
            res[key].append(m[key])
    return {m: (float(np.mean(res[m])),
                float(np.percentile(res[m], 2.5)),
                float(np.percentile(res[m], 97.5))) for m in METRICS}


# %%
# =============================================================================
# CELL 4 — PIPELINE FACTORIES (identical to PGIMER)
# =============================================================================
def make_classifier(clf_name):
    if clf_name == "RF":
        return RandomForestClassifier(n_estimators=200, max_depth=4, min_samples_split=4,
                                      min_samples_leaf=2, max_features="sqrt",
                                      class_weight="balanced", random_state=SEED, n_jobs=-1)
    if clf_name == "SVM_Lin":
        return SVC(kernel="linear", C=1, class_weight="balanced",
                   probability=True, random_state=SEED)
    if clf_name == "LR":
        return LogisticRegression(C=1, max_iter=3000, class_weight="balanced",
                                  solver="lbfgs", random_state=SEED)
    if clf_name == "LDA":
        return LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
    if clf_name == "XGBoost":
        return XGBClassifier(n_estimators=50, max_depth=2, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8, reg_lambda=5,
                             objective="binary:logistic", eval_metric="logloss",
                             random_state=SEED, n_jobs=-1)
    raise ValueError(f"Unknown classifier: {clf_name}")


def make_pipeline_for_grid(clf_name):
    pipe = Pipeline([
        ("scaler", RobustScaler()),
        ("sel", SelectKBest(score_func=f_classif, k=K_VALUES[0])),
        ("clf", make_classifier(clf_name)),
    ])
    return GridSearchCV(pipe, {"sel__k": K_VALUES}, cv=LeaveOneOut(),
                        scoring="accuracy", refit=True, n_jobs=-1, error_score=0.0)


# ---------------- 1D-CNN (same architecture as PGIMER run) ----------------
def set_all_seeds(seed):
    np.random.seed(seed)
    tf.random.set_seed(seed)


def build_cnn(input_dim):
    model = Sequential([
        Reshape((input_dim, 1), input_shape=(input_dim,)),
        Conv1D(32, 3, activation="relu", padding="same"),
        BatchNormalization(),
        MaxPooling1D(2),
        Conv1D(64, 3, activation="relu", padding="same"),
        BatchNormalization(),
        MaxPooling1D(2),
        Flatten(),
        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return model


def cnn_fit_predict(X_tr, y_tr, X_te, k, seed):
    """Scaler and ANOVA selector fitted on X_tr only (no leakage)."""
    set_all_seeds(seed)
    scaler, selector = RobustScaler(), SelectKBest(f_classif, k=k)
    Xtr = selector.fit_transform(scaler.fit_transform(X_tr), y_tr)
    Xte = selector.transform(scaler.transform(X_te))
    tf.keras.backend.clear_session()
    model = build_cnn(k)
    cw = {0: len(y_tr) / (2 * np.sum(y_tr == 0)), 1: len(y_tr) / (2 * np.sum(y_tr == 1))}
    model.fit(Xtr, y_tr, epochs=100, batch_size=16, validation_split=0.1,
              callbacks=[EarlyStopping(monitor="val_loss", patience=10,
                                       restore_best_weights=True, verbose=0)],
              class_weight=cw, verbose=0)
    return model.predict(Xte, verbose=0).flatten()


def cnn_select_k(X_tr, y_tr, seed):
    """Inner stratified 5-fold CV on the outer training fold, scored by accuracy."""
    inner = StratifiedKFold(n_splits=CNN_INNER_FOLDS, shuffle=True, random_state=42)
    scores = {}
    for k in K_VALUES:
        accs = []
        for itr, iva in inner.split(X_tr, y_tr):
            prob = cnn_fit_predict(X_tr[itr], y_tr[itr], X_tr[iva], k, seed)
            accs.append(accuracy_score(y_tr[iva], (prob >= 0.5).astype(int)))
        scores[k] = np.mean(accs)
    return max(scores, key=scores.get)


def run_mcnemar(pred_dict, Y):
    if not HAS_STATSMODELS:
        return {}
    out = {}
    for a, b in combinations(pred_dict.keys(), 2):
        pa, pb = np.asarray(pred_dict[a]), np.asarray(pred_dict[b])
        n01 = int(np.sum((pa == Y) & (pb != Y)))
        n10 = int(np.sum((pa != Y) & (pb == Y)))
        r = mcnemar_test([[0, n01], [n10, 0]], exact=True)
        out[(a, b)] = {"p_value": float(r.pvalue), "significant": r.pvalue < 0.05}
    return out

print("Pipeline factories created.")


# %%
# =============================================================================
# CELL 5 — REPEATED NESTED CROSS-VALIDATION MAIN LOOP
# =============================================================================
all_nested_results = []

for outer_seed in OUTER_SEEDS:
    print("\n" + "#" * 80)
    print(f"OUTER SEED = {outer_seed}")
    print("#" * 80)

    outer_cv = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=outer_seed)
    all_preds = {c: np.zeros(len(Y), dtype=int) for c in CLF_NAMES}
    all_probs = {c: np.zeros(len(Y), dtype=float) for c in CLF_NAMES}
    best_k_log = {c: [] for c in CLF_NAMES}

    for fold_idx, (tr, te) in enumerate(outer_cv.split(X, Y), 1):
        print(f"Fold {fold_idx:02d}/{N_OUTER_FOLDS} | Train={len(tr)} | Test={len(te)} | "
              f"Test dist={dict(Counter(Y[te].tolist()))}")
        for c in CLASSICAL:
            gs = make_pipeline_for_grid(c)
            gs.fit(X[tr], Y[tr])
            best_k_log[c].append(gs.best_params_["sel__k"])
            all_preds[c][te] = gs.predict(X[te])
            all_probs[c][te] = gs.predict_proba(X[te])[:, 1]

        if RUN_CNN:
            k_cnn = cnn_select_k(X[tr], Y[tr], outer_seed)
            prob = cnn_fit_predict(X[tr], Y[tr], X[te], k_cnn, outer_seed)
            best_k_log["1D-CNN"].append(k_cnn)
            all_probs["1D-CNN"][te] = prob
            all_preds["1D-CNN"][te] = (prob >= 0.5).astype(int)
            print(f"   1D-CNN done (k={k_cnn})")

    seed_metrics = {c: compute_metrics(Y, all_preds[c], all_probs[c]) for c in CLF_NAMES}
    seed_ci = {c: bootstrap_ci(Y, all_preds[c], all_probs[c]) for c in CLF_NAMES}

    all_nested_results.append({
        "outer_seed": outer_seed,
        "all_preds": {c: all_preds[c].copy() for c in CLF_NAMES},
        "all_probs": {c: all_probs[c].copy() for c in CLF_NAMES},
        "metrics": seed_metrics,
        "ci": seed_ci,
        "best_k_log": {c: list(best_k_log[c]) for c in CLF_NAMES},
        "mcnemar": run_mcnemar(all_preds, Y),
        "Y": Y.copy(),
    })

    print("\nSeed summary:")
    for c in CLF_NAMES:
        m = seed_metrics[c]
        print(f"{c:<10} | Acc={m['accuracy']*100:6.2f}% | AUC={m['roc_auc']:.3f} | "
              f"Sens={m['sensitivity']*100:6.2f}% | Spec={m['specificity']*100:6.2f}% | "
              f"TP={m['TP']} TN={m['TN']} FP={m['FP']} FN={m['FN']}")

with open(f"{OUT_PREFIX}_nested_cv_results.pkl", "wb") as f:
    pickle.dump(all_nested_results, f)
print(f"\nSaved: {OUT_PREFIX}_nested_cv_results.pkl")


# %%
# =============================================================================
# CELL 6 — SUMMARY: MEAN ± SD ACROSS SEEDS + BOOTSTRAP 95% CI
# =============================================================================
rows = []
print("\n" + "=" * 110)
print("MODMA WITHIN-DATASET NESTED CV — MEAN ± SD ACROSS SEEDS  [bootstrap 95% CI]")
print("AUC on 0–1 scale; other metrics in %")
print("=" * 110)
for c in CLF_NAMES:
    row = {"Classifier": c}
    line = f"{c:<10}"
    for m in METRICS:
        vals = np.array([r["metrics"][c][m] for r in all_nested_results])
        lo = np.mean([r["ci"][c][m][1] for r in all_nested_results])
        hi = np.mean([r["ci"][c][m][2] for r in all_nested_results])
        if m == "roc_auc":
            txt = f"{vals.mean():.3f} ± {vals.std(ddof=1):.3f} [{lo:.3f}–{hi:.3f}]"
        else:
            txt = f"{vals.mean()*100:.2f} ± {vals.std(ddof=1)*100:.2f} [{lo*100:.2f}–{hi*100:.2f}]"
        row[m] = txt
        line += f" | {m}: {txt}"
    rows.append(row)
    print(line)

summary_df = pd.DataFrame(rows)
summary_df.to_csv(f"{OUT_PREFIX}_summary.csv", index=False)
print(f"\nSaved: {OUT_PREFIX}_summary.csv")
print("Report SVM_Lin (primary SHIELD classifier) and 1D-CNN in Table 9; all models in the supplement.")


# %%
# =============================================================================
# CELL 7 — BEST k DISTRIBUTION
# =============================================================================
print("\n" + "=" * 80)
print(f"BEST k DISTRIBUTION ({len(OUTER_SEEDS)} seeds × {N_OUTER_FOLDS} folds)")
print("=" * 80)
for c in CLF_NAMES:
    ks = Counter(k for r in all_nested_results for k in r["best_k_log"][c])
    print(f"{c:<10} " + "  ".join(f"k={k}:{ks.get(k, 0):>2}" for k in K_VALUES)
          + f"   most common: k={ks.most_common(1)[0][0]}")


# %%
# =============================================================================
# CELL 8 — McNEMAR SUMMARY (optional)
# =============================================================================
if HAS_STATSMODELS:
    print("\n" + "=" * 80)
    print("McNEMAR TEST ACROSS SEEDS (p < 0.05 = different error pattern)")
    print("=" * 80)
    for pair in combinations(CLF_NAMES, 2):
        ps = [r["mcnemar"][pair]["p_value"] for r in all_nested_results]
        sig = sum(r["mcnemar"][pair]["significant"] for r in all_nested_results)
        print(f"{pair[0] + ' vs ' + pair[1]:<25} mean p = {np.mean(ps):.4f}   significant seeds: {sig}/{len(OUTER_SEEDS)}")

# Note: no biomarker ranking here — biomarkers are identified on PGIMER only.

SHIELD — REPEATED NESTED CV WITHIN MODMA
Classifiers : ['RF', 'SVM_Lin', 'LR', 'LDA', 'XGBoost', '1D-CNN']
k values    : [10, 15, 20, 25, 30]
Outer folds : 10   Seeds : [42, 0, 1, 7, 99]
Raw shape: (53, 316)
Labels: MDD (1) = 24 | HC (0) = 29  (expected 24 MDD, 29 HC)
Mapped MODMA features to PGIMER naming: 314 features
Final feature matrix: X = (53, 314), Y = (53,)
Pipeline factories created.

################################################################################
OUTER SEED = 42
################################################################################
Fold 01/10 | Train=47 | Test=6 | Test dist={1: 3, 0: 3}
   1D-CNN done (k=25)
Fold 02/10 | Train=47 | Test=6 | Test dist={1: 3, 0: 3}
   1D-CNN done (k=25)
Fold 03/10 | Train=47 | Test=6 | Test dist={1: 3, 0: 3}
   1D-CNN done (k=10)
Fold 04/10 | Train=48 | Test=5 | Test dist={1: 2, 0: 3}
   1D-CNN done (k=10)
Fold 05/10 | Train=48 | Test=5 | Test dist={1: 2, 0: 3}
   1D-CNN done (k=20)
Fold 06/10 | Train=48 | Test=5 | T

In [1]:
import sys
import sklearn
import shap
import scipy
import statsmodels

print(f"Python version: {sys.version.split()[0]}")
print(f"scikit-learn version: {sklearn.__version__}")
print(f"SHAP version: {shap.__version__}")
print(f"SciPy version: {scipy.__version__}")
print(f"statsmodels version: {statsmodels.__version__}")

Python version: 3.9.18
scikit-learn version: 1.3.2
SHAP version: 0.46.0
SciPy version: 1.12.0
statsmodels version: 0.12.2
